# SQLSage on Google Colab (GPU)

Pipeline: **install → (optional ZeroGPU) → Space liveness → Phase 14 check → rollouts → GRPO training → plots**.

1. **Runtime → Change runtime type → GPU** (T4 for 1.5B 4-bit smoke; **A100** Colab = faster; see **Option A vs Option B** in **Cell 2** for HF ZeroGPU *remote* Gradio).
2. **Colab secrets** (optional): `WANDB_API_KEY`, `SQLSAGE_ENV_URL` (OpenEnv / FastAPI **base** URL for rollouts and training), `HF_TOKEN` (Hub upload + private Gradio), `WANDB_ENTITY`, `SQLSAGE_HF_SPACE_URL`, `SQLSAGE_WANDB_RUN_ID`, and optionally **`HF_ZEROGPU_SPACE`** + **`SQLSAGE_TRY_ZEROGPU=1`** to exercise **Cell 3** (Gradio client). If you only set `SQLSAGE_ENV_URL`, **Cell 4** copies it to `SQLSAGE_HF_SPACE_URL` when unset.
3. **Repo** is cloned to `/content/SQLSage` and the notebook `cd`s into `sqlsage-env/` when `pyproject.toml` is there.
4. The **SQLSage** Space must serve **`GET /health`** and **`POST /reset`**. Cold-start can take many minutes. This is *not* the same as a ZeroGPU Gradio demo; see Cell 2.
5. **Phase 14 (prompts + verification):** run **Cell 5** — `test_phase14_integration.py` checks `rewrite_patterns`, `prompt_builder`, `training_verifier` (mock W&B), and few-shot vs baseline length. The **Full pipeline** step may **SKIP** in Colab (no TPC-H Postgres). For GRPO, `train.py` uses **real Space observations** + `make_prompt` unless `SQLSAGE_TRAIN_PLAIN_PROMPT=1`. **Cell 7** sets **`SQLSAGE_MAX_SEQ_LENGTH=2048`**.
6. **Cell 7** default hyperparams are T4-friendly. Set `SQLSAGE_NUM_TRAIN_EPOCHS=3` (or higher) for a full run.

**Team tooling in-repo** (see **Cell 9**): `python -m sqlsage.run help`, `python -m sqlsage.status_checker` / `--json`, `python -m sqlsage.generate_cheatsheets`. Optional: `sqlsage.training_verifier` when you have W&B + model + DB.


In [ ]:
# Cell 1 — GPU check, clone repo, install package

# Optional: os.environ["SQLSAGE_GIT_URL"] = "https://github.com/YOU/SQLSage.git"

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q trl openenv-core wandb transformers datasets accelerate huggingface_hub requests "psycopg2-binary"

import os, pathlib, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 (or A100).")

print("Device:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)

REPO = pathlib.Path("/content/SQLSage")
URL = os.environ.get("SQLSAGE_GIT_URL", "https://github.com/neelshingavi/SQLSage.git")
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)
else:
    print("Reusing", REPO)

if (REPO / "pyproject.toml").is_file():
    os.chdir(REPO)
elif (REPO / "sqlsage-env" / "pyproject.toml").is_file():
    os.chdir(REPO / "sqlsage-env")
else:
    raise FileNotFoundError("pyproject.toml not found at /content/SQLSage or .../sqlsage-env")

ROOT = pathlib.Path().resolve()
print("Working directory:", ROOT)
# training + plots + run extras (rich, python-dotenv) for `sqlsage.run` and dashboard-style helpers
%pip install -q -e '.[training,plots,run]'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
pathlib.Path("results").mkdir(exist_ok=True)
print("OK: sqlsage package ready.")


### Option A vs Option B (compute)

- **Option A — This notebook’s runtime GPU (T4 / A100 in Colab):** Used for Unsloth + GRPO. Pro: full control. Con: T4 is tight on context; upgrade runtime if OOM.
- **Option B — Hugging Face [ZeroGPU](https://huggingface.co/spaces/enzostvs/zero-gpu-spaces) via `gradio_client`:** Some community **Gradio** Spaces queue work on an **H200-class** GPU when a job runs. You call them from Colab with `gradio_client` as a **remote** backend (inference, image jobs, or text generation **if** the Space exposes a Gradio API). Pro: very strong GPUs when the queue grants them. Con: **queue latency**, **IP rate limits**, and APIs **break** if the Space author changes the app.

**SQLSage note:** Your **`SQLSAGE_ENV_URL`** OpenEnv is usually a **normal HTTP/JSON** API, **not** Gradio `predict()`. This notebook’s rollouts and `train.py` still talk to that URL. The optional **Cells 2–3** below are for **other** Gradio Spaces (e.g. a heavy LLM or vision model), not a substitute for the env Space.

**Pro tip:** On any Space, open the page, scroll to **“Use via API”**, and copy the exact `api_name` and argument types into your `client.predict(...)` call.

In [ ]:
# Cell 3 — (Optional) ZeroGPU: Gradio `gradio_client` — only runs if SQLSAGE_TRY_ZEROGPU=1
# Secrets: HF_ZEROGPU_SPACE, SQLSAGE_TRY_ZEROGPU, HF_TOKEN (Colab, or re-run after Cell 4)
%pip install -q "gradio_client>=1.0"

import os
from gradio_client import Client

try:
    from google.colab import userdata
    for key in ("HF_ZEROGPU_SPACE", "SQLSAGE_TRY_ZEROGPU"):
        try:
            v = userdata.get(key)
        except Exception:  # noqa: BLE001
            v = None
        if v and not (os.environ.get(key) or "").strip():
            os.environ[key] = v.strip()
except ImportError:
    pass

os.environ.setdefault("SQLSAGE_TRY_ZEROGPU", "0")
SPACE = (os.environ.get("HF_ZEROGPU_SPACE", "") or "").strip()

if os.environ.get("SQLSAGE_TRY_ZEROGPU", "0").strip() not in ("1", "true", "yes") or not SPACE:
    print(
        "ZeroGPU: skipped. To try: set Secret HF_ZEROGPU_SPACE=org/SpaceName, "
        "set SQLSAGE_TRY_ZEROGPU=1, re-run this cell, then use client.view_api() and client.predict(...)."
    )
else:
    tok = (os.environ.get("HF_TOKEN", "") or "").strip() or None
    print("Connecting to Gradio Space:", SPACE)
    client = Client(SPACE, hf_token=tok) if tok else Client(SPACE)
    print("API surface (endpoints):")
    try:
        client.view_api()
    except Exception as e:  # noqa: BLE001
        print("view_api failed (check gradio_client version):", e)
    # Example (replace with the Space's own signature from "Use via API"):
    # result = client.predict("hello", api_name="/chat")
    # print(result)
    print("Done. Paste predict() from the Space 'Use via API' docs; avoid hammering the queue (rate limits).")

In [ ]:
# Cell 4 — Colab secrets, W&B, Space liveness ( /health, /docs, or /reset )

import os, sys, time, requests, wandb

try:
    from google.colab import userdata
    for key in (
        "WANDB_API_KEY",
        "WANDB_ENTITY",
        "SQLSAGE_ENV_URL",
        "SQLSAGE_HF_SPACE_URL",
        "SQLSAGE_WANDB_RUN_ID",
        "HF_TOKEN",
        "HF_ZEROGPU_SPACE",
        "SQLSAGE_TRY_ZEROGPU",
    ):
        try:
            v = userdata.get(key)
        except Exception:
            v = None
        if v and not os.environ.get(key, "").strip():
            os.environ[key] = v.strip()
except ImportError:
    pass

if os.environ.get("HF_TOKEN", "").strip():
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"].strip()

os.environ.setdefault("SQLSAGE_ENV_URL", "https://adity00-sqlsage-env.hf.space")
os.environ.setdefault("WANDB_PROJECT", "sqlsage-grpo")

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
# sqlsage.status_checker uses this name; default to the same Space URL as training/rollouts
if not (os.environ.get("SQLSAGE_HF_SPACE_URL") or "").strip():
    os.environ["SQLSAGE_HF_SPACE_URL"] = base
print("SQLSAGE_ENV_URL =", base)
print("SQLSAGE_HF_SPACE_URL =", os.environ.get("SQLSAGE_HF_SPACE_URL", ""))

if os.environ.get("WANDB_API_KEY", "").strip():
    try:
        wandb.login()
        print("W&B: logged in")
    except Exception as e:
        print("W&B login warning:", e)
else:
    os.environ["WANDB_MODE"] = "offline"
    print("W&B: offline (add Secret WANDB_API_KEY to sync to cloud)")

def space_ready(url: str) -> None:
    read_timeout = int(os.environ.get("SQLSAGE_HEALTH_READ_TIMEOUT", "120"))
    max_wait = int(os.environ.get("SQLSAGE_HEALTH_MAX_WAIT", "600"))
    poll = int(os.environ.get("SQLSAGE_HEALTH_POLL_S", "8"))
    deadline = time.time() + max_wait
    n = 0
    while time.time() < deadline:
        n += 1
        for path in ("/health", "/docs", "/openapi.json"):
            try:
                r = requests.get(url + path, timeout=read_timeout)
                if r.status_code == 200:
                    print("Space OK:", path, "->", r.status_code)
                    r2 = requests.post(url + "/reset", json={}, timeout=180)
                    print("/reset", r2.status_code, (r2.text or "")[:200])
                    if r2.status_code == 200:
                        ob = (r2.json() or {}).get("observation", {})
                        print("task_level", ob.get("task_level"), "step", ob.get("step_count"))
                    return
            except requests.RequestException as e:
                print("Try", n, path, e)
        time.sleep(poll)
    raise RuntimeError("Space not reachable. Open the .hf.space URL in a browser, wait, re-run.")

space_ready(base)
print("— Ready for rollouts & training —")


In [ ]:
# Cell 5 — Phase 14 integration (rewrite_patterns, prompt_builder, training_verifier)
# Run after Cell 4 (secrets). Exits non-zero if a core test FAILs. Full-pipeline/Postgres may SKIP in Colab.
import subprocess, sys, pathlib

root = pathlib.Path().resolve()
script = root / "test_phase14_integration.py"
if not script.is_file():
    print("Skip: test_phase14_integration.py not found — pull latest SQLSage / sqlsage-env.")
else:
    subprocess.check_call([sys.executable, str(script)], cwd=root)


In [ ]:
# Cell 4 — Rollout smoke + baseline JSONL (identity policy) → W&B

import os, sys, subprocess, pathlib

base = os.environ["SQLSAGE_ENV_URL"].rstrip("/")
proj = os.environ.get("WANDB_PROJECT", "sqlsage-grpo")
name = os.environ.get("WANDB_RUN_NAME_ROLLOUT", "colab-baseline")
ep_smoke = int(os.environ.get("SQLSAGE_ROLLOUT_SMOKE_EPISODES", "5"))
ep_base = int(os.environ.get("SQLSAGE_ROLLOUT_BASELINE_EPISODES", "50"))
envp = os.environ.copy()

def run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args, env=envp, cwd=pathlib.Path().resolve())

run_cmd([sys.executable, "scripts/rollout_wandb.py", "--episodes", str(ep_smoke), "--base-url", base, "--project", proj, "--run-name", f"{name}-smoke", "--out-jsonl", "results/smoke.jsonl"])
run_cmd([sys.executable, "scripts/rollout_wandb.py", "--episodes", str(ep_base), "--policy", "identity", "--base-url", base, "--project", proj, "--run-name", f"{name}-baseline", "--out-jsonl", "results/baseline.jsonl"])
if pathlib.Path("results").is_dir():
    for p in sorted(pathlib.Path("results").glob("*.jsonl")):
        print(p, p.stat().st_size, "bytes")


In [ ]:
# Cell 7 — Full GRPO (train.run_training) — T4 + Phase 14 prompts (see train.py)

import os
# Longer context: make_prompt + REWRITE_PATTERN_HINTS + few-shots (Phase 14). T4: 2048; A100: 4096 ok.
os.environ["SQLSAGE_MAX_SEQ_LENGTH"] = os.environ.get("SQLSAGE_MAX_SEQ_LENGTH", "2048")
os.environ["SQLSAGE_MAX_COMPLETION_LENGTH"] = "256"
os.environ["SQLSAGE_NUM_GENERATIONS"] = "4"
os.environ["SQLSAGE_PER_DEVICE_TRAIN_BATCH_SIZE"] = "2"
os.environ["SQLSAGE_GRADIENT_ACCUMULATION_STEPS"] = "8"
# 1 = smoke, 3 = full run
os.environ["SQLSAGE_NUM_TRAIN_EPOCHS"] = os.environ.get("SQLSAGE_NUM_TRAIN_EPOCHS", "1")
os.environ["SQLSAGE_GRPO_TEMPERATURE"] = "0.9"
os.environ["SQLSAGE_OUTPUT_DIR"] = "sqlsage-grpo"
os.environ["SQLSAGE_MERGED_DIR"] = "sqlsage-grpo-merged"
os.environ["SQLSAGE_MODEL_NAME"] = os.environ.get("SQLSAGE_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
os.environ["WANDB_RUN_NAME"] = os.environ.get("WANDB_GRPO_RUN_NAME", "colab-grpo-t4")
# Real observations + make_prompt (default). Set to 1 for a short generic prompt smoke only.
os.environ.setdefault("SQLSAGE_TRAIN_PLAIN_PROMPT", "0")
# Dataset size for precomputed reset() prompts (default 256 in train.py)
os.environ.setdefault("SQLSAGE_TRAIN_DATASET_SIZE", "256")

assert os.environ.get("SQLSAGE_ENV_URL", "").strip(), "Set SQLSAGE_ENV_URL (Colab Secret or above)"

from train import run_training
print(run_training())


In [ ]:
# Cell 6 — Plots, artifact paths, optional download
# Same as local: `python -m sqlsage.run p2-plots` (from repo root)

!python plots/generate_plots.py
!ls -la plots/*.png 2>/dev/null; ls -la sqlsage-grpo/ sqlsage-grpo-merged/ 2>/dev/null; true
# from google.colab import files; files.download("plots/reward_curve.png")
# !python scripts/push_model_to_hub.py --folder ./sqlsage-grpo-merged --repo-id USER/sqlsage-merged
# `python -m sqlsage.run p3-benchmark` needs a local DB — run on your dev machine, not in Colab
print("LoRA: ./sqlsage-grpo | merged 16bit: ./sqlsage-grpo-merged")


In [ ]:
# Cell 9 — (Optional) team utilities: status checks, cheat sheets, alias list
# Re-run any time. Does not start training. Exit codes may be != 0 if a milestone is "red" — that is expected.

import os, subprocess, sys, pathlib

ROOT = pathlib.Path().resolve()
os.environ.setdefault("SQLSAGE_HF_SPACE_URL", (os.environ.get("SQLSAGE_ENV_URL", "") or "").rstrip("/"))
print("SQLSAGE_HF_SPACE_URL (for /reset check):", os.environ.get("SQLSAGE_HF_SPACE_URL", "(not set)"))
if (os.environ.get("SQLSAGE_WANDB_RUN_ID") or "").strip():
    print("SQLSAGE_WANDB_RUN_ID set (entity/project/run) — full W&B gate checks enabled")

# JSON milestone + reward/plot/git checks
subprocess.run(
    [sys.executable, "-m", "sqlsage.status_checker", "--json"],
    cwd=ROOT,
)
print("---")

# Printable one-page MD + HTML cheat sheets under ./cheatsheets/
subprocess.run(
    [sys.executable, "-m", "sqlsage.generate_cheatsheets"],
    cwd=ROOT,
)
print("Cheat sheets:", ROOT / "cheatsheets")
print("---")

# Short list of one-command aliases (p3-gpu, p2-reward, status, …)
subprocess.run(
    [sys.executable, "-m", "sqlsage.run", "help"],
    cwd=ROOT,
)